In [9]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. IMPORT DATASET
# ==========================================

df = pd.read_csv("Telecom_Customer_Churn.csv")

print("Dataset Imported Successfully!")
print(df.head())


# ==========================================
# 2. EXPLORE DATASET
# ==========================================

print("\nFirst 5 Records:")
print(df.head())

print("\nDataset Information:")
print(df.info())

print("\nDataset Shape:")
print(df.shape)

print("\nStatistical Summary:")
print(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Records:")
print(df.duplicated().sum())


# ==========================================
# 3. HANDLE MISSING VALUES
# ==========================================

# Convert TotalCharges to numeric if present
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"], errors="coerce"
    )

# Identify numerical and categorical columns
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include=["object", "category"]).columns

# Fill numerical missing values with median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values with mode
for col in cat_cols:
    if not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing Values After Cleaning:")
print(df.isnull().sum())


# ==========================================
# 4. REMOVE DUPLICATE RECORDS
# ==========================================

df = df.drop_duplicates()

print("\nShape After Removing Duplicates:")
print(df.shape)


# ==========================================
# 5. HANDLE INCONSISTENT DATA
# ==========================================

# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Remove extra spaces from categorical values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Standardize Yes/No values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].replace({
        "yes": "Yes",
        "YES": "Yes",
        "no": "No",
        "NO": "No"
    })

print("\nInconsistent Data Standardized!")


# ==========================================
# 6. CONVERT COLUMNS TO CORRECT DATA TYPES
# ==========================================

if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"], errors="coerce"
    )

if "SeniorCitizen" in df.columns:
    df["SeniorCitizen"] = df["SeniorCitizen"].astype("category")

print("\nData Types:")
print(df.dtypes)


# ==========================================
# 7. IDENTIFY AND HANDLE OUTLIERS
# ==========================================

num_cols = df.select_dtypes(include=np.number).columns

for col in num_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Cap outliers
    df[col] = df[col].clip(lower, upper)

print("\nOutliers Handled Using IQR Method!")


# ==========================================
# 8. FEATURE ENGINEERING
# ==========================================

# Create average monthly spending
if "TotalCharges" in df.columns and "tenure" in df.columns:

    df["AverageMonthlySpend"] = (
        df["TotalCharges"] /
        df["tenure"].replace(0, np.nan)
    )

    df["AverageMonthlySpend"] = df["AverageMonthlySpend"].fillna(0)

# Create tenure groups
if "tenure" in df.columns:

    df["TenureGroup"] = pd.cut(
        df["tenure"],
        bins=[-1, 12, 36, np.inf],
        labels=["New", "Medium", "Loyal"]
    )

print("\nNew Features Created!")
print(df.head())


# ==========================================
# 9. NORMALIZE / SCALE DATA
# ==========================================

# Identify numerical columns
num_cols = df.select_dtypes(include=np.number).columns

# Apply StandardScaler
scaler = StandardScaler()

df[num_cols] = scaler.fit_transform(df[num_cols])

print("\nNumerical Data Scaled Successfully!")


# ==========================================
# 10. SPLIT DATA INTO TRAINING AND TESTING
# ==========================================

# Check target column
if "Churn" not in df.columns:
    raise ValueError("Churn column not found in dataset!")

# Separate target and features
y = df["Churn"].map({"Yes": 1, "No": 0})

if y.isnull().any():
    raise ValueError("Churn column must contain Yes/No values.")

X = df.drop("Churn", axis=1)

# Convert categorical columns into numerical columns
X = pd.get_dummies(X, drop_first=True, dtype=int)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

print("\nTraining and Testing Split Completed!")


# ==========================================
# 11. EXPORT CLEANED DATASET
# ==========================================

# Combine encoded features and target
cleaned_df = X.copy()
cleaned_df["Churn"] = y

# Export cleaned dataset
cleaned_df.to_csv(
    "Cleaned_Telecom_Customer_Churn.csv",
    index=False
)

# Export training and testing datasets
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

Dataset Imported Successfully!
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV Strea

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load and explore dataset
df = pd.read_csv("telecom_churn_dataset.csv")
print(df.head())
df.info()
print(df.isnull().sum())

# 3. Remove duplicates
df = df.drop_duplicates()

# 4. Handle missing values
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())


# 5. Handle outliers using IQR
num_cols = [
    "age", "tenure_months", "data_usage_gb",
    "total_day_minutes", "total_eve_minutes",
    "total_night_minutes", "total_intl_minutes",
    "total_day_calls", "total_eve_calls",
    "total_night_calls", "total_intl_calls",
    "customer_service_calls", "monthly_charges", "total_charges"
]

for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = df[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)


# 6. Feature engineering
df["tenure_group"] = pd.cut(
    df["tenure_months"],
    bins=[0, 12, 36, 60, 72],
    labels=["0-12", "13-36", "37-60", "61-72"]
)

# 7. Encode target and categorical columns
df["churn"] = df["churn"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
df = pd.get_dummies(df.drop(columns=["customer_id"]), drop_first=True)

# 8. Save cleaned dataset
df.to_csv("Telecom_Customer_Churn_Cleaned.csv", index=False)

# 9. Split into training and testing data
X = df.drop(columns=["churn"])
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 10. Scale numerical features
scaler = StandardScaler()
X_train.loc[:, num_cols] = scaler.fit_transform(X_train[num_cols])
X_test.loc[:, num_cols] = scaler.transform(X_test[num_cols])

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Cleaned dataset saved!")

   customer_id  gender  age  senior_citizen           state city_tier  \
0  CUST-000001    Male   26               0  Andhra Pradesh    Tier 3   
1  CUST-000002  Female   53               0      Tamil Nadu    Tier 1   
2  CUST-000003  Female   23               0       Rajasthan    Tier 3   
3  CUST-000004  Female   41               0       Karnataka    Tier 2   
4  CUST-000005    Male   43               0          Kerala    Tier 3   

   tenure_months   contract_type plan_type     data_plan  ... total_day_calls  \
0             19  Month-to-Month  Postpaid          1 GB  ...              14   
1             61        One Year   Prepaid          5 GB  ...              82   
2             28        One Year  Postpaid          5 GB  ...              17   
3             56  Month-to-Month   Prepaid          2 GB  ...               3   
4              1        Two Year  Postpaid  No Data Plan  ...               3   

  total_eve_calls total_night_calls total_intl_calls customer_service_call

C:\Users\athar\AppData\Local\Temp\ipykernel_12184\2302332877.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.1485159  -1.46976498 -1.34965143 ...  1.71324416  0.03165443
  1.23278995]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_train.loc[:, num_cols] = scaler.fit_transform(X_train[num_cols])
C:\Users\athar\AppData\Local\Temp\ipykernel_12184\2302332877.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 0.61007556  0.46551874  0.65826116 ...  0.36914752 -0.01633733
  0.80281798]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_train.loc[:, num_cols] = scaler.fit_transform(X_train[num_cols])
C:\Users\athar\AppData\Local\Temp\ipykernel_12184\2302332877.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a f